In [ ]:
import subprocess
import os
import csv
import pandas as pd
from bayes_opt import BayesianOptimization
from pathlib import Path
import numpy as np

In [16]:
#VOT config
#alphaWalk,alphaBike,alphaCarDriver,alphaCarPassenger,alphaBus,alphaTrain,betaChangesTransport,betaCostLow,betaCostMed,betaCostHigh,votCommuteWalk,votCommuteBike,votCommuteCar,votCommuteBus,votCommuteTrain,votOtherWalk,votOtherBike,votOtherCar,votOtherBus,votOtherTrain,carCostKm,ptCostKm,ptBaseCost,weightAccessEgress
defaultAlpha_bounds  = (-5,5)
default_beta_bounds = (-1, -0.001)
pbounds = {'alphaWalk' :(0,0) ,
           'alphaBike' : defaultAlpha_bounds,
           'alphaCarDriver' : defaultAlpha_bounds,
           'alphaCarPassenger' : defaultAlpha_bounds,
           'alphaBus' : defaultAlpha_bounds,
           'alphaTrain' : defaultAlpha_bounds,
           'betaChangesTransport': default_beta_bounds,
           'betaCostLow':(-0.4,-0.2) ,
           'betaCostMed': (-0.2,-0.2),
           'betaCostHigh': (-0.2,-0.01),
           'weightWalk': (1,2),
           'weightWait': (1,2),
           'weightFeeder': (1,2), 
           'weightVotCosts' : (1,1),
           'weightTangibleCosts' : (1,1)
           } 

In [ ]:
def writeVotConfigFile(
        alphaWalk,
        alphaBike,
        alphaCarDriver,
        alphaCarPassenger,
        alphaBus,
        alphaTrain,
        betaChangesTransport,
        betaCostLow,
        betaCostMed,
        betaCostHigh,
        weightWalk,
        weightWait,
        weightFeeder,
        weightVotCosts,
        weightTangibleCosts,
        filename) -> int:
    data = {'alphaWalk' : alphaWalk,
                'alphaBike' :alphaBike,
                'alphaCarDriver': alphaCarDriver,
                'alphaCarPassenger':alphaCarPassenger,
                'alphaBus':alphaBus,
                'alphaTrain' :alphaTrain,
                'betaChangesTransport':betaChangesTransport,
                'betaCostLow' :betaCostLow, # betaCostLow,
                'betaCostMed' : betaCostMed,
                'betaCostHigh': betaCostHigh, #betaCostHigh,
                'votCommuteWalk': 15.89,
                'votCommuteBike': 10.17,
                'votCommuteCar': 10.78,
                'votCommuteBus': 7.62,
                'votCommuteTrain': 12.05,
                'votOtherWalk': 11.76,
                'votOtherBike': 10.43,
                'votOtherCar':9.60,
                'votOtherBus':6.66,
                'votOtherTrain':8.64,
                'carCostKm':0.25,
                'ptCostKm':0.187,
                'ptBaseCost':1.08,
                'weightWalk': weightWalk,# 2.0,
                'weightWait': weightWait, # 2.5,
                'weightFeeder': weightFeeder, # 2.0,
                'weightVotCosts' : weightVotCosts,
                'weightTangibleCosts' : weightTangibleCosts
                }
    if Path(filename).exists():
        config = pd.read_csv(filename)
        new_entry = pd.DataFrame([data])

        row_count = len(config) #Length off by one, so new index

        new_entry.to_csv(filename,mode='a',header=(not os.path.exists(filename)) , index = None)

        return row_count
    else:
        cols = [
        'alphaWalk', 'alphaBike', 'alphaCarDriver', 'alphaCarPassenger', 'alphaBus', 
        'alphaTrain', 'betaChangesTransport', 'betaCostLow', 'betaCostMed', 'betaCostHigh',
        'votCommuteWalk', 'votCommuteBike', 'votCommuteCar', 'votCommuteBus', 'votCommuteTrain',
        'votOtherWalk', 'votOtherBike', 'votOtherCar', 'votOtherBus', 'votOtherTrain',
        'carCostKm', 'ptCostKm', 'ptBaseCost', 'weightWalk','weightWait','weightFeeder', 'weightVotCosts',
        'weightTangibleCosts'
            ]
        
        df = pd.DataFrame([data], columns=cols)


        df.to_csv(filename, index=None)
        return 0
    
    

### Prepare the Java command

In [ ]:
def callSimulation(
        alphaWalk,
        alphaBike,
        alphaCarDriver,
        alphaCarPassenger,
        alphaBus,
        alphaTrain,
        betaChangesTransport,
        betaCostLow,
        betaCostMed,
        betaCostHigh,
        weightWalk,
        weightWait,
        weightFeeder,
        weightVotCosts,
        weightTangibleCosts
        ):
    
    filename = "../7-simulation-Sim-2APL/src/main/resources/baseline_parameterset/vot_parameterset_bayesian.csv"
    parameter_set_index = writeVotConfigFile(
        alphaWalk,
        alphaBike,
        alphaCarDriver,
        alphaCarPassenger,
        alphaBus,
        alphaTrain,
        betaChangesTransport,
        betaCostLow,
        betaCostMed,
        betaCostHigh,
        weightWalk,
        weightWait,
        weightFeeder,
        weightVotCosts,
        weightTangibleCosts,
        filename
        )

    config = "--config src/main/resources/config_DHZW_full.toml"
    parameters_path = "--parameter_file src/main/resources/baseline_parameterset/vot_parameterset_bayesian.csv"
    output_path = f'--output_file src/main/resources/baseline_parameterset/output_proportionsCOPY.csv'
    arg = f'--parameterset_index {parameter_set_index}'
    java_folder_path = '7-simulation-Sim-2APL'


    # set current directory the folder of Sim2APL so I can execute the jar with the correct paths
    if (os.path.basename(os.getcwd()) != java_folder_path):
        new_directory = os.path.join(os.getcwd(), "../", java_folder_path)
        new_directory = new_directory.replace('\\', '/')
        os.chdir(new_directory)

    full_command = f'java -cp target/sim2apl-dhzw-simulation-1.0-SNAPSHOT-jar-with-dependencies.jar main.java.nl.uu.iss.ga.Simulation {config} {output_path} {parameters_path} {arg}'

    try:
        output = subprocess.check_output(
            full_command, stderr=subprocess.STDOUT, universal_newlines=True)
        return -float(output)
    except subprocess.CalledProcessError as e:
        print(f"Java program exited with non-zero return code: {e.returncode}")
        print(f"Error message: {e.output}")
        exit(1)
    return -999999999999 #an egregiously bad option.

In [19]:
import os
import subprocess


jdk_11_path = r"C:\Program Files\Java\jdk-11"

os.environ["JAVA_HOME"] = jdk_11_path
bin_path = os.path.join(jdk_11_path, "bin")
os.environ["PATH"] = bin_path + os.pathsep + os.environ.get("PATH", "")

# 4. Verification
try:
    result = subprocess.run(["java", "-version"], capture_output=True, text=True, check=True)
    print("Java Version Output:\n", result.stderr) 
except subprocess.CalledProcessError as e:
    print(f"Error: {e}")
except FileNotFoundError:
    print("Java executable not found in the updated PATH.")

Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



In [20]:
optimizer = BayesianOptimization(
    f=callSimulation,
    pbounds=pbounds,
    random_state=1,
)

In [21]:

optimizer.maximize(
    init_points=20,
    n_iter=20,
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaCo... | betaCo... | betaCo... | weight... | weight... | weight... | weight... | weight... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.35290 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.813926 | -0.330887 | -0.2      | -0.097624 | 1.4191945 | 1.6852195 | 1.2044522 | 1.0       | 1.0       |
| 2         | -15.05085 | 0.0       | -0.826951 | 0.5868982 | -3.596130 | -3.018985 | 3.0074456 | -0.032706 | -0.337315 | -0.2      | -0.033486 | 1.8946066 | 1.0850442 | 1.0390547 | 1.0       | 1.0       |
| 3         | -8.126619 | 0.0       | -0.788923 | 4.5788953 | 0.3316528 | 1.9187711 | -1.844843 | -0.314185 | -0.233074 | -0.2      | -0.057472 | 1.9888610 | 1.7481656 | 1.2804

In [22]:
print(optimizer.max)

{'target': np.float64(-6.088040291057547), 'params': {'alphaWalk': np.float64(0.0), 'alphaBike': np.float64(-0.3632452623043161), 'alphaCarDriver': np.float64(5.0), 'alphaCarPassenger': np.float64(1.1613773097094078), 'alphaBus': np.float64(0.39350554838539314), 'alphaTrain': np.float64(-0.9067449854921228), 'betaChangesTransport': np.float64(-0.001), 'betaCostLow': np.float64(-0.4), 'betaCostMed': np.float64(-0.2), 'betaCostHigh': np.float64(-0.2), 'weightWalk': np.float64(1.0), 'weightWait': np.float64(2.0), 'weightFeeder': np.float64(2.0), 'weightVotCosts': np.float64(1.0), 'weightTangibleCosts': np.float64(1.0)}}


In [24]:
results = []
for i in range(0,10):
    results.append(- callSimulation(**optimizer.max['params']))

In [25]:
results

[6.097963324521286,
 6.095916450718109,
 6.09668817733307,
 6.122999376644635,
 6.081611338336337,
 6.094696671810338,
 6.090480798105547,
 6.092589186324862,
 6.1054048883954275,
 6.105248636300754]

In [28]:
np.mean(results)

np.float64(6.098359884849037)

In [29]:
np.std(results)

np.float64(0.01049965558180599)